Horribly messy point data from EMLID and TOPCON receivers

In [30]:
from pathlib import Path
import zipfile
import shutil
import re
import csv

# ------------------------------------------------------------
# SOURCE DIRECTORIES — READ ONLY
# ------------------------------------------------------------

EMLID_DIR = Path(
    r"D:\My Drive\BOP_OCTC_2025\2026 RTK Files\EMLID"
)

TOPCON_DIR = Path(
    r"D:\My Drive\BOP_OCTC_2025\2026 RTK Files\TOPCON\SHAPEFILE"
)

# ------------------------------------------------------------
# NEW OUTPUT DIRECTORY
# ------------------------------------------------------------

OUTPUT_DIR = Path(
    r"D:\My Drive\BOP_OCTC_2025\2026 RTK Files\CORRECTED_SHAPEFILES"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("EMLID :", EMLID_DIR)
print("TOPCON:", TOPCON_DIR)
print("OUTPUT:", OUTPUT_DIR)

EMLID : D:\My Drive\BOP_OCTC_2025\2026 RTK Files\EMLID
TOPCON: D:\My Drive\BOP_OCTC_2025\2026 RTK Files\TOPCON\SHAPEFILE
OUTPUT: D:\My Drive\BOP_OCTC_2025\2026 RTK Files\CORRECTED_SHAPEFILES


In [2]:
def clean_zip_stem(zip_path):
    """
    Convert ZIP filename to standardized dataset basename.

    Examples
    --------
    20260527_mid_17.shp.zip
        -> 20260527_mid_17

    20260616_high_36_rtk.zip
        -> 20260616_high_36

    20260506_low_41.zip
        -> 20260506_low_41
    """

    name = zip_path.name

    # Remove final .zip
    if name.lower().endswith(".zip"):
        name = name[:-4]

    # Some Emlid exports are named *.shp.zip
    if name.lower().endswith(".shp"):
        name = name[:-4]

    # RTK is provenance, not part of the plot/node identifier
    if name.lower().endswith("_rtk"):
        name = name[:-4]

    return name


def is_date_named_shapefile(filename):
    """
    Topcon whole-node shapefiles appear to begin with YYYYMMDD_.

    Examples accepted:
        20260506_low_41.shp
        20260713_A4.shp

    Examples rejected:
        Low_41_0.0.shp
        low_41_11.0.shp
    """

    stem = Path(filename).stem

    return bool(
        re.match(r"^\d{8}_", stem)
    )

In [3]:
def inventory_zip_directory(folder, system):
    records = []

    for zip_path in sorted(folder.glob("*.zip")):

        standardized_name = clean_zip_stem(zip_path)

        try:
            with zipfile.ZipFile(zip_path, "r") as z:

                shp_files = [
                    Path(name).name
                    for name in z.namelist()
                    if name.lower().endswith(".shp")
                ]

                records.append({
                    "system": system,
                    "zip": zip_path.name,
                    "standardized_name": standardized_name,
                    "shapefiles_found": shp_files,
                    "n_shapefiles": len(shp_files)
                })

        except zipfile.BadZipFile:

            records.append({
                "system": system,
                "zip": zip_path.name,
                "standardized_name": standardized_name,
                "shapefiles_found": ["BAD ZIP"],
                "n_shapefiles": 0
            })

    return records


inventory = (
    inventory_zip_directory(EMLID_DIR, "EMLID")
    +
    inventory_zip_directory(TOPCON_DIR, "TOPCON")
)


for rec in inventory:
    print("\n" + rec["system"], rec["zip"])
    print("  target:", rec["standardized_name"])
    print("  SHPs:", rec["shapefiles_found"])


EMLID 20260527_mid_17.shp.zip
  target: 20260527_mid_17
  SHPs: ['Points.shp']

EMLID 20260528_low_47.shp.zip
  target: 20260528_low_47
  SHPs: ['Points.shp']

EMLID 20260601_high_29.shp.zip
  target: 20260601_high_29
  SHPs: ['Points.shp']

EMLID 20260602_low_51.shp.zip
  target: 20260602_low_51
  SHPs: ['Points.shp']

EMLID 20260603_low_60.shp.zip
  target: 20260603_low_60
  SHPs: ['Points.shp']

EMLID 20260603_low_68.shp.zip
  target: 20260603_low_68
  SHPs: ['Points.shp']

EMLID 20260604_mid_14.shp.zip
  target: 20260604_mid_14
  SHPs: ['Points.shp']

EMLID 20260608_mid_22.shp.zip
  target: 20260608_mid_22
  SHPs: ['Points.shp']

EMLID 20260609_mid_20.shp.zip
  target: 20260609_mid_20
  SHPs: ['Points.shp']

EMLID 20260610_mid_26.shp.zip
  target: 20260610_mid_26
  SHPs: ['Points.shp']

EMLID 20260611_low_69.shp.zip
  target: 20260611_low_69
  SHPs: ['Points.shp']

EMLID 20260611_low_80.shp.zip
  target: 20260611_low_80
  SHPs: ['Points.shp']

EMLID 20260615_high_32.shp.zip
  targ

In [4]:
def copy_emlid_zip(zip_path, output_dir):

    target_stem = clean_zip_stem(zip_path)

    copied = []

    with zipfile.ZipFile(zip_path, "r") as z:

        members = z.namelist()

        # Anything belonging to the Points shapefile family
        point_files = [
            member for member in members
            if Path(member).stem.lower() == "points"
        ]

        if not any(
            Path(member).suffix.lower() == ".shp"
            for member in point_files
        ):
            raise RuntimeError(
                f"No Points.shp found in {zip_path.name}"
            )

        for member in point_files:

            source_name = Path(member).name
            extension = Path(source_name).suffix.lower()

            target_path = output_dir / f"{target_stem}{extension}"

            # Do not silently overwrite anything
            if target_path.exists():
                raise FileExistsError(
                    f"Output already exists: {target_path}"
                )

            with z.open(member) as src, open(target_path, "wb") as dst:
                shutil.copyfileobj(src, dst)

            copied.append(target_path.name)

    return copied

In [5]:
def copy_topcon_zip(zip_path, output_dir):

    target_stem = clean_zip_stem(zip_path)

    copied = []

    with zipfile.ZipFile(zip_path, "r") as z:

        members = z.namelist()

        # Find date-prefixed SHP files.
        whole_plot_shps = [
            member
            for member in members
            if (
                member.lower().endswith(".shp")
                and
                is_date_named_shapefile(Path(member).name)
            )
        ]

        if len(whole_plot_shps) == 0:
            raise RuntimeError(
                f"No date-named whole-plot shapefile found in "
                f"{zip_path.name}"
            )

        if len(whole_plot_shps) > 1:
            raise RuntimeError(
                f"More than one candidate whole-plot shapefile "
                f"found in {zip_path.name}: "
                f"{whole_plot_shps}"
            )

        selected_shp = whole_plot_shps[0]
        selected_stem = Path(selected_shp).stem

        # Grab every component belonging to that shapefile
        family = [
            member
            for member in members
            if Path(member).stem == selected_stem
        ]

        for member in family:

            source_name = Path(member).name
            extension = Path(source_name).suffix.lower()

            target_path = output_dir / f"{target_stem}{extension}"

            if target_path.exists():
                raise FileExistsError(
                    f"Output already exists: {target_path}"
                )

            with z.open(member) as src, open(target_path, "wb") as dst:
                shutil.copyfileobj(src, dst)

            copied.append(target_path.name)

    return copied

In [6]:
manifest = []


# ============================================================
# EMLID
# ============================================================

for zip_path in sorted(EMLID_DIR.glob("*.zip")):

    print(f"EMLID  : {zip_path.name}")

    try:

        copied = copy_emlid_zip(
            zip_path,
            OUTPUT_DIR
        )

        manifest.append({
            "system": "EMLID",
            "source_zip": zip_path.name,
            "output_stem": clean_zip_stem(zip_path),
            "status": "COPIED",
            "files_copied": "; ".join(copied)
        })

    except Exception as e:

        print(f"    ERROR: {e}")

        manifest.append({
            "system": "EMLID",
            "source_zip": zip_path.name,
            "output_stem": clean_zip_stem(zip_path),
            "status": f"ERROR: {e}",
            "files_copied": ""
        })


# ============================================================
# TOPCON
# ============================================================

for zip_path in sorted(TOPCON_DIR.glob("*.zip")):

    print(f"TOPCON : {zip_path.name}")

    try:

        copied = copy_topcon_zip(
            zip_path,
            OUTPUT_DIR
        )

        manifest.append({
            "system": "TOPCON",
            "source_zip": zip_path.name,
            "output_stem": clean_zip_stem(zip_path),
            "status": "COPIED",
            "files_copied": "; ".join(copied)
        })

    except Exception as e:

        print(f"    ERROR: {e}")

        manifest.append({
            "system": "TOPCON",
            "source_zip": zip_path.name,
            "output_stem": clean_zip_stem(zip_path),
            "status": f"ERROR: {e}",
            "files_copied": ""
        })


print("\nFinished.")

EMLID  : 20260527_mid_17.shp.zip
EMLID  : 20260528_low_47.shp.zip
EMLID  : 20260601_high_29.shp.zip
EMLID  : 20260602_low_51.shp.zip
EMLID  : 20260603_low_60.shp.zip
EMLID  : 20260603_low_68.shp.zip
EMLID  : 20260604_mid_14.shp.zip
EMLID  : 20260608_mid_22.shp.zip
EMLID  : 20260609_mid_20.shp.zip
EMLID  : 20260610_mid_26.shp.zip
EMLID  : 20260611_low_69.shp.zip
EMLID  : 20260611_low_80.shp.zip
EMLID  : 20260615_high_32.shp.zip
EMLID  : 20260615_mid_10.shp.zip
EMLID  : 20260616_mid_4.shp.zip
EMLID  : 20260630_mid_18.shp.zip
EMLID  : 20260630_mid_8.shp.zip
EMLID  : 20260701_mid_19.shp.zip
EMLID  : 20260707_mid_5.shp.zip
EMLID  : 20260708_high_34.shp.zip
EMLID  : 20260709_mid_2.shp.zip
EMLID  : 20260713_A1.shp.zip
EMLID  : 20260714_B3.shp.zip
EMLID  : 20260715_A5.shp.zip
EMLID  : 20260716_C1.shp.zip
EMLID  : 20260720_C3.shp.zip
EMLID  : 20260721_D5.shp.zip
EMLID  : 20260722_D3.shp.zip
EMLID  : 20260723_F2.shp.zip
EMLID  : 20260804_high_31.shp.zip
TOPCON : 20260506_low_41.zip
TOPCON : 2026

In [7]:
manifest_csv = OUTPUT_DIR / "shapefile_copy_manifest.csv"

with open(
    manifest_csv,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "system",
            "source_zip",
            "output_stem",
            "status",
            "files_copied"
        ]
    )

    writer.writeheader()
    writer.writerows(manifest)


print(f"Manifest written to:\n{manifest_csv}")

Manifest written to:
D:\My Drive\BOP_OCTC_2025\2026 RTK Files\CORRECTED_SHAPEFILES\shapefile_copy_manifest.csv


In [8]:
shapefiles = sorted(
    OUTPUT_DIR.glob("*.shp")
)

print(f"Corrected shapefiles: {len(shapefiles)}\n")

for shp in shapefiles:
    print(shp.name)

Corrected shapefiles: 58

20260506_low_41.shp
20260511_low_62.shp
20260512_low_63.shp
20260513_low_50.shp
20260518_low_49.shp
20260519_high_35.shp
20260521_mid_16.shp
20260526_mid_12.shp
20260527_high_33.shp
20260527_low_77.shp
20260527_mid_17.shp
20260528_low_47.shp
20260601_high_29.shp
20260602_low_43.shp
20260602_low_51.shp
20260603_low_59.shp
20260603_low_60.shp
20260603_low_68.shp
20260603_low_76.shp
20260604_mid_14.shp
20260604_mid_21.shp
20260608_mid_22.shp
20260609_mid_20.shp
20260610_mid_26.shp
20260611_low_69.shp
20260611_low_80.shp
20260615_high_32.shp
20260615_low_74.shp
20260615_mid_10.shp
20260616_high_36.shp
20260616_low_65.shp
20260616_mid_4.shp
20260630_high_38.shp
20260630_mid_18.shp
20260630_mid_8.shp
20260701_mid_19.shp
20260707_mid_5.shp
20260708_high_34.shp
20260709_low_42rtk.shp
20260709_mid_2.shp
20260713_A1.shp
20260713_A4.shp
20260714_A3.shp
20260714_B3.shp
20260715_A5.shp
20260715_A6.shp
20260716_B5.shp
20260716_C1.shp
20260720_C2.shp
20260720_C3.shp
20260721

Now that we have corrected shapefiles, we need to label them appropriately - the TOPCON does not have good point metadata internally, so we use the CSVs to bring them up to spec. The following cells need to be run on the arcpy kernel 3.11.11 instead of 3.13

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import arcpy
import shutil
import re
import os

CORRECTED_DIR = Path(
    r"N:\Data02\projects-active\BOPclassification_2025\2026 Data\CORRECTED_SHAPEFILES"
)

TOPCON_CSV_DIR = Path(
    r"D:\My Drive\BOP_OCTC_2025\2026 RTK Files\TOPCON\CSV"
)

REBUILT_DIR = Path(
    r"N:\Data02\projects-active\BOPclassification_2025\2026 Data\TOPCON_REBUILT"
)

REBUILT_DIR.mkdir(parents=True, exist_ok=True)

# These are RTK-derived coordinates. Matching should be extremely tight.
MATCH_TOLERANCE_M = 0.10

print("Corrected SHPs:", CORRECTED_DIR)
print("TOPCON CSVs:    ", TOPCON_CSV_DIR)
print("Output:         ", REBUILT_DIR)

Corrected SHPs: N:\Data02\projects-active\BOPclassification_2025\2026 Data\CORRECTED_SHAPEFILES
TOPCON CSVs:     D:\My Drive\BOP_OCTC_2025\2026 RTK Files\TOPCON\CSV
Output:          N:\Data02\projects-active\BOPclassification_2025\2026 Data\TOPCON_REBUILT


In [2]:
def normalize_stem(path_or_name):
    """
    Examples:
        20260527_high_33_rtk.csv -> 20260527_high_33
        20260709_low_42rtk.csv   -> 20260709_low_42
        20260506_low_41.csv      -> 20260506_low_41
    """

    stem = Path(path_or_name).stem

    stem = re.sub(
        r"_?rtk$",
        "",
        stem,
        flags=re.IGNORECASE
    )

    return stem

In [3]:
pairs = []

for csv_path in sorted(TOPCON_CSV_DIR.glob("*.csv")):

    stem = normalize_stem(csv_path)
    shp_path = CORRECTED_DIR / f"{stem}.shp"

    pairs.append({
        "stem": stem,
        "csv": csv_path,
        "shp": shp_path,
        "shp_exists": shp_path.exists()
    })


for p in pairs:
    print(
        f"{p['stem']:<28} "
        f"CSV=YES   "
        f"SHP={'YES' if p['shp_exists'] else 'MISSING'}"
    )

20260506_low_41              CSV=YES   SHP=YES
20260511_low_62              CSV=YES   SHP=YES
20260512_low_63              CSV=YES   SHP=YES
20260513_low_50              CSV=YES   SHP=YES
20260518_low_49              CSV=YES   SHP=YES
20260519_high_35             CSV=YES   SHP=YES
20260521_mid_16              CSV=YES   SHP=YES
20260526_mid_12              CSV=YES   SHP=YES
20260527_high_33             CSV=YES   SHP=YES
20260527_low_77              CSV=YES   SHP=YES
20260602_low_43              CSV=YES   SHP=YES
20260603_low_59              CSV=YES   SHP=YES
20260603_low_76              CSV=YES   SHP=YES
20260604_mid_21              CSV=YES   SHP=YES
20260615_low_74              CSV=YES   SHP=YES
20260616_high_36             CSV=YES   SHP=YES
20260616_low_65              CSV=YES   SHP=YES
20260630_high_38             CSV=YES   SHP=YES
20260709_low_42              CSV=YES   SHP=MISSING
20260713_A4                  CSV=YES   SHP=YES
20260714_A3                  CSV=YES   SHP=YES
20260715_

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np
import arcpy
import shutil
import re

from scipy.optimize import linear_sum_assignment


# ------------------------------------------------------------------
# EXISTING INPUTS
# ------------------------------------------------------------------

CORRECTED_DIR = Path(
    r"N:\Data02\projects-active\BOPclassification_2025\2026 Data\CORRECTED_SHAPEFILES"
)

TOPCON_CSV_DIR = Path(
    r"D:\My Drive\BOP_OCTC_2025\2026 RTK Files\TOPCON\CSV"
)


# ------------------------------------------------------------------
# NEW OUTPUT DIRECTORY
# ------------------------------------------------------------------

TOPCON_REBUILT_DIR = Path(
    r"N:\Data02\projects-active\BOPclassification_2025\2026 Data\TOPCON_REBUILT"
)

TOPCON_REBUILT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------------
# SOURCE CRS
# ------------------------------------------------------------------
# NAD83(2011) / UTM Zone 11N

SOURCE_SR = arcpy.SpatialReference(6340)


# ------------------------------------------------------------------
# MATCHING TOLERANCE
# ------------------------------------------------------------------
# Existing Topcon shapefile geometry and CSV coordinates should be
# essentially identical. Start conservatively.

MATCH_TOLERANCE_M = 0.25


print("Corrected SHPs :", CORRECTED_DIR)
print("TOPCON CSVs    :", TOPCON_CSV_DIR)
print("Rebuilt output :", TOPCON_REBUILT_DIR)
print("Source CRS     :", SOURCE_SR.name)

Corrected SHPs : N:\Data02\projects-active\BOPclassification_2025\2026 Data\CORRECTED_SHAPEFILES
TOPCON CSVs    : D:\My Drive\BOP_OCTC_2025\2026 RTK Files\TOPCON\CSV
Rebuilt output : N:\Data02\projects-active\BOPclassification_2025\2026 Data\TOPCON_REBUILT
Source CRS     : NAD_1983_2011_UTM_Zone_11N


In [5]:
def read_topcon_csv(csv_path):
    """
    Read a headerless Topcon CSV.

    Source columns:
        Plot_ID
        X
        Y
        Z

    NOTE:
    The Topcon convention in these files is:
        X ~ 4,800,000
        Y ~   550,000

    Therefore for GIS geometry:
        geometry X = CSV Y
        geometry Y = CSV X

    But the source attributes remain named X, Y, Z.
    """

    df = pd.read_csv(
        csv_path,
        header=None,
        names=["Plot_ID", "X", "Y", "Z"],
        usecols=[0, 1, 2, 3]
    )

    df["Plot_ID"] = (
        df["Plot_ID"]
        .astype(str)
        .str.strip()
    )

    for col in ["X", "Y", "Z"]:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    return df

In [6]:
test_csv = TOPCON_CSV_DIR / "20260506_low_41.csv"

test = read_topcon_csv(test_csv)

display(test.head(10))

print("Rows:", len(test))
print("\nNulls:")
print(test.isna().sum())

,Plot_ID,X,Y,Z
0,100,4.808041e+06,549902.4736,861.4262
1,20260506_low_41_rtk,4.808013e+06,549919.9190,862.0000
2,Low_41_0.0,4.808041e+06,549902.4594,861.4018
3,Low_41_0.1,4.808050e+06,549904.4535,860.3251
4,Low_41_0.2,4.808038e+06,549912.2720,860.9529
5,Low_41_0.3,4.808031e+06,549900.4516,862.4511
6,Low_41_0.4,4.808043e+06,549892.7230,861.6401
7,Low_41_11.0,4.808291e+06,549903.7720,855.4050
8,Low_41_11.1,4.808301e+06,549905.2985,854.8662
9,Low_41_11.2,4.808289e+06,549913.5866,854.6138


Rows: 42

Nulls:
Plot_ID    0
X          0
Y          0
Z          0
dtype: int64


In [7]:
def copy_shapefile_family(source_shp, output_dir):
    """
    Copy every file belonging to a shapefile family.

    Example:
        node.shp
        node.dbf
        node.shx
        node.prj
        node.clf
        ...

    Originals remain untouched.
    """

    source_shp = Path(source_shp)
    stem = source_shp.stem

    source_family = sorted(
        source_shp.parent.glob(f"{stem}.*")
    )

    if not source_family:
        raise FileNotFoundError(
            f"No shapefile family found for {source_shp}"
        )

    copied = []

    for src in source_family:

        dst = output_dir / src.name

        if dst.exists():
            raise FileExistsError(
                f"Output already exists:\n{dst}"
            )

        shutil.copy2(src, dst)
        copied.append(dst)

    return copied

In [8]:
def get_shapefile_points(shp_path):
    """
    Return OID and geometry coordinates for every point.
    """

    rows = []

    with arcpy.da.SearchCursor(
        str(shp_path),
        ["OID@", "SHAPE@XY"]
    ) as cursor:

        for oid, xy in cursor:

            rows.append({
                "OID": oid,
                "Geom_X": float(xy[0]),
                "Geom_Y": float(xy[1])
            })

    return pd.DataFrame(rows)

In [9]:
def match_shapefile_to_csv(
    shp_path,
    csv_df,
    tolerance=0.25
):
    """
    One-to-one spatial matching between existing shapefile geometry
    and Topcon CSV coordinates.

    Geometry:
        ArcGIS X <- CSV Y
        ArcGIS Y <- CSV X
    """

    shp_df = get_shapefile_points(shp_path)

    csv_valid = (
        csv_df
        .dropna(subset=["X", "Y"])
        .copy()
        .reset_index(drop=True)
    )

    if len(shp_df) == 0:
        raise RuntimeError(
            f"No features found in {shp_path}"
        )

    if len(csv_valid) == 0:
        raise RuntimeError(
            f"No valid coordinates in CSV for {shp_path}"
        )


    # --------------------------------------------------------------
    # Geometry coordinates from CSV
    # --------------------------------------------------------------

    csv_geom_x = csv_valid["Y"].to_numpy(dtype=float)
    csv_geom_y = csv_valid["X"].to_numpy(dtype=float)

    shp_x = shp_df["Geom_X"].to_numpy(dtype=float)
    shp_y = shp_df["Geom_Y"].to_numpy(dtype=float)


    # --------------------------------------------------------------
    # Pairwise Euclidean distances
    # --------------------------------------------------------------

    dx = shp_x[:, None] - csv_geom_x[None, :]
    dy = shp_y[:, None] - csv_geom_y[None, :]

    distance_matrix = np.sqrt(dx**2 + dy**2)


    # --------------------------------------------------------------
    # Globally optimal one-to-one assignment
    # --------------------------------------------------------------

    shp_indices, csv_indices = linear_sum_assignment(
        distance_matrix
    )


    matches = []

    for si, ci in zip(shp_indices, csv_indices):

        distance = float(
            distance_matrix[si, ci]
        )

        matches.append({
            "OID": int(shp_df.iloc[si]["OID"]),
            "csv_index": int(ci),
            "distance_m": distance,
            "within_tolerance": distance <= tolerance
        })


    match_df = pd.DataFrame(matches)

    return shp_df, csv_valid, match_df

In [10]:
test_shp = CORRECTED_DIR / "20260506_low_41.shp"
test_csv = TOPCON_CSV_DIR / "20260506_low_41.csv"

test_df = read_topcon_csv(test_csv)

shp_pts, csv_pts, matches = match_shapefile_to_csv(
    test_shp,
    test_df,
    tolerance=MATCH_TOLERANCE_M
)


print("Shapefile features:", len(shp_pts))
print("CSV records:       ", len(csv_pts))
print("Matched:           ", len(matches))

print(
    "Maximum distance:  ",
    matches["distance_m"].max()
)

print(
    "Median distance:   ",
    matches["distance_m"].median()
)

print(
    "Outside tolerance: ",
    (~matches["within_tolerance"]).sum()
)

display(
    matches.sort_values(
        "distance_m",
        ascending=False
    ).head(10)
)

Shapefile features: 41
CSV records:        42
Matched:            41
Maximum distance:   6.265277434928335e-05
Median distance:    4.2551574059132166e-05
Outside tolerance:  0


,OID,csv_index,distance_m,within_tolerance
17,17,23,0.000063,True
33,33,29,0.000061,True
20,20,26,0.000059,True
19,19,25,0.000057,True
10,10,21,0.000056,True
1,1,2,0.000056,True
4,4,5,0.000055,True
31,31,27,0.000053,True
40,40,16,0.000052,True
12,12,8,0.000051,True


In [11]:
def add_rebuilt_fields(shp_path):
    """
    Add clean Topcon source fields if absent.
    """

    existing = {
        f.name.lower()
        for f in arcpy.ListFields(str(shp_path))
    }

    if "plot_id" not in existing:
        arcpy.management.AddField(
            str(shp_path),
            "Plot_ID",
            "TEXT",
            field_length=100
        )

    if "x" not in existing:
        arcpy.management.AddField(
            str(shp_path),
            "X",
            "DOUBLE"
        )

    if "y" not in existing:
        arcpy.management.AddField(
            str(shp_path),
            "Y",
            "DOUBLE"
        )

    if "z" not in existing:
        arcpy.management.AddField(
            str(shp_path),
            "Z",
            "DOUBLE"
        )

    if "sourcecsv" not in existing:
        arcpy.management.AddField(
            str(shp_path),
            "SourceCSV",
            "TEXT",
            field_length=100
        )

In [12]:
def populate_rebuilt_attributes(
    shp_path,
    csv_path,
    tolerance=0.25
):
    """
    Match copied shapefile geometry to CSV coordinates and write:

        Plot_ID
        X
        Y
        Z
        SourceCSV

    Returns QA information.
    """

    csv_df = read_topcon_csv(csv_path)

    shp_df, csv_valid, matches = match_shapefile_to_csv(
        shp_path,
        csv_df,
        tolerance=tolerance
    )


    # --------------------------------------------------------------
    # Refuse to silently populate questionable matches
    # --------------------------------------------------------------

    bad_matches = matches[
        ~matches["within_tolerance"]
    ]

    if len(bad_matches) > 0:

        raise RuntimeError(
            f"{Path(shp_path).name}: "
            f"{len(bad_matches)} matches exceed "
            f"{tolerance} m tolerance. "
            "No attributes written."
        )


    # OID -> CSV record
    lookup = {}

    for _, match in matches.iterrows():

        oid = int(match["OID"])
        ci = int(match["csv_index"])

        src = csv_valid.iloc[ci]

        lookup[oid] = {
            "Plot_ID": src["Plot_ID"],
            "X": src["X"],
            "Y": src["Y"],
            "Z": src["Z"]
        }


    # --------------------------------------------------------------
    # Write attributes
    # --------------------------------------------------------------

    fields = [
        "OID@",
        "Plot_ID",
        "X",
        "Y",
        "Z",
        "SourceCSV"
    ]

    updated = 0

    with arcpy.da.UpdateCursor(
        str(shp_path),
        fields
    ) as cursor:

        for row in cursor:

            oid = row[0]

            if oid not in lookup:
                continue

            src = lookup[oid]

            row[1] = src["Plot_ID"]
            row[2] = src["X"]
            row[3] = src["Y"]
            row[4] = src["Z"]
            row[5] = Path(csv_path).name

            cursor.updateRow(row)

            updated += 1


    return {
        "features": len(shp_df),
        "csv_rows": len(csv_valid),
        "matched": len(matches),
        "updated": updated,
        "max_distance_m": matches["distance_m"].max(),
        "median_distance_m": matches["distance_m"].median()
    }

In [17]:
# --------------------------------------------------------------
# RESET LOG
# --------------------------------------------------------------

rebuild_log = []


# --------------------------------------------------------------
# COPY FUNCTION
# --------------------------------------------------------------

def copy_shapefile_family(source_shp, output_dir):
    """
    Copy only useful shapefile components.

    Existing derivative outputs with the same stem are removed first.
    Source files are never modified.

    .clf files are ignored.
    """

    source_shp = Path(source_shp)
    output_dir = Path(output_dir)

    stem = source_shp.stem

    allowed_extensions = {
        ".shp",
        ".shx",
        ".dbf",
        ".prj",
        ".cpg",
        ".sbn",
        ".sbx",
        ".xml"
    }

    # ----------------------------------------------------------
    # Remove previous DERIVATIVE output family
    # ----------------------------------------------------------

    for existing in output_dir.glob(f"{stem}.*"):

        # Explicitly ignore stray .clf files
        if existing.suffix.lower() == ".clf":
            continue

        try:
            existing.unlink()

        except PermissionError:
            raise RuntimeError(
                f"Cannot replace {existing.name}; "
                "the file appears to be locked in ArcGIS Pro."
            )

    # ----------------------------------------------------------
    # Copy useful source components only
    # ----------------------------------------------------------

    source_family = [
        f for f in source_shp.parent.glob(f"{stem}.*")
        if f.suffix.lower() in allowed_extensions
    ]

    if not any(f.suffix.lower() == ".shp" for f in source_family):
        raise FileNotFoundError(
            f"No shapefile found for {source_shp}"
        )

    copied = []

    for src in source_family:

        dst = output_dir / src.name
        shutil.copy2(src, dst)

        copied.append(dst)

    return copied


# --------------------------------------------------------------
# BULK REBUILD
# --------------------------------------------------------------

for p in pairs:

    stem = p["stem"]
    csv_path = p["csv"]
    source_shp = p["shp"]

    # ----------------------------------------------------------
    # low_42 has no source shapefile; handle separately
    # ----------------------------------------------------------

    if not p["shp_exists"]:

        print(
            f"NO SHP — defer to CSV reconstruction: {stem}"
        )

        rebuild_log.append({
            "stem": stem,
            "method": "CSV -> geometry",
            "status": "DEFERRED"
        })

        continue


    rebuilt_shp = (
        TOPCON_REBUILT_DIR /
        f"{stem}.shp"
    )

    print(f"\nRebuilding: {stem}")


    try:

        # ------------------------------------------------------
        # Copy original geometry
        # ------------------------------------------------------

        copy_shapefile_family(
            source_shp,
            TOPCON_REBUILT_DIR
        )


        # ------------------------------------------------------
        # Add clean fields
        # ------------------------------------------------------

        add_rebuilt_fields(
            rebuilt_shp
        )


        # ------------------------------------------------------
        # Restore attributes from CSV
        # ------------------------------------------------------

        stats = populate_rebuilt_attributes(
            rebuilt_shp,
            csv_path,
            tolerance=MATCH_TOLERANCE_M
        )


        # ------------------------------------------------------
        # Successful log entry
        # ------------------------------------------------------

        rebuild_log.append({
            "stem": stem,
            "method": "SHP + CSV",
            "status": "OK",
            "features": stats["features"],
            "csv_rows": stats["csv_rows"],
            "matched": stats["matched"],
            "updated": stats["updated"],
            "max_distance_m": stats["max_distance_m"],
            "median_distance_m": stats["median_distance_m"]
        })


        print(
            f"  features: {stats['features']}"
        )

        print(
            f"  CSV rows: {stats['csv_rows']}"
        )

        print(
            f"  matched:  {stats['matched']}"
        )

        print(
            f"  updated:  {stats['updated']}"
        )

        print(
            f"  max dist: "
            f"{stats['max_distance_m']:.6f} m"
        )


    except Exception as e:

        print(f"  ERROR: {e}")

        rebuild_log.append({
            "stem": stem,
            "method": "SHP + CSV",
            "status": "ERROR",
            "error": str(e)
        })


print("\nBulk Topcon rebuild complete.")


Rebuilding: 20260506_low_41
  ERROR: ERROR 000852: Cannot add field SourceCSV to 20260506_low_41
Failed to execute (AddField).


Rebuilding: 20260511_low_62
  features: 51
  CSV rows: 51
  matched:  51
  updated:  51
  max dist: 0.000058 m

Rebuilding: 20260512_low_63
  features: 36
  CSV rows: 36
  matched:  36
  updated:  36
  max dist: 0.000066 m

Rebuilding: 20260513_low_50
  features: 46
  CSV rows: 46
  matched:  46
  updated:  46
  max dist: 0.000068 m

Rebuilding: 20260518_low_49
  features: 37
  CSV rows: 37
  matched:  37
  updated:  37
  max dist: 0.000067 m

Rebuilding: 20260519_high_35
  ERROR: ERROR 000852: Cannot add field SourceCSV to 20260519_high_35
Failed to execute (AddField).


Rebuilding: 20260521_mid_16
  features: 52
  CSV rows: 52
  matched:  52
  updated:  52
  max dist: 0.000060 m

Rebuilding: 20260526_mid_12
  features: 41
  CSV rows: 41
  matched:  41
  updated:  41
  max dist: 0.000064 m

Rebuilding: 20260527_high_33
  features: 38
  CSV rows: 38
  matche

In [15]:
low42_csv = (
    TOPCON_CSV_DIR /
    "20260709_low_42rtk.csv"
)

low42_output = (
    TOPCON_REBUILT_DIR /
    "20260709_low_42.shp"
)


# --------------------------------------------------------------
# Read source CSV
# --------------------------------------------------------------

low42 = read_topcon_csv(
    low42_csv
)

bad_xy = low42[
    low42["X"].isna() |
    low42["Y"].isna()
]

if len(bad_xy) > 0:

    print("Rows with invalid X/Y:")
    display(bad_xy)

    raise RuntimeError(
        "low_42 contains invalid coordinates. "
        "Nothing created."
    )


print("low_42 records:", len(low42))
display(low42.head())

low_42 records: 48


,Plot_ID,X,Y,Z
0,20260709_low_42_0.0,4.762300e+06,592397.6458,893.0541
1,20260709_low_42_0.1,4.762309e+06,592400.3628,892.8853
2,20260709_low_42_0.2,4.762297e+06,592406.9936,892.9276
3,20260709_low_42_0.3,4.762290e+06,592395.0799,893.1809
4,20260709_low_42_0.4,4.762302e+06,592387.8014,893.2644


In [18]:
qa = pd.DataFrame(rebuild_log)

display(
    qa.sort_values("stem")
)

print("\n-----------------------------------")
print("TOPCON REBUILD SUMMARY")
print("-----------------------------------")

print(
    "Successful nodes:",
    (qa["status"] == "OK").sum()
)

print(
    "Failed nodes:",
    (qa["status"] != "OK").sum()
)

print(
    "Total rebuilt features:",
    qa.loc[
        qa["status"] == "OK",
        "features"
    ].sum()
)

,stem,method,status,error,features,csv_rows,matched,updated,max_distance_m,median_distance_m
0,20260506_low_41,SHP + CSV,ERROR,ERROR 000852: Cannot add field SourceCSV to 20...,NaN,NaN,NaN,NaN,NaN,NaN
1,20260511_low_62,SHP + CSV,OK,NaN,51.0,51.0,51.0,51.0,0.000058,0.000042
2,20260512_low_63,SHP + CSV,OK,NaN,36.0,36.0,36.0,36.0,0.000066,0.000042
3,20260513_low_50,SHP + CSV,OK,NaN,46.0,46.0,46.0,46.0,0.000068,0.000039
4,20260518_low_49,SHP + CSV,OK,NaN,37.0,37.0,37.0,37.0,0.000067,0.000040
5,20260519_high_35,SHP + CSV,ERROR,ERROR 000852: Cannot add field SourceCSV to 20...,NaN,NaN,NaN,NaN,NaN,NaN
6,20260521_mid_16,SHP + CSV,OK,NaN,52.0,52.0,52.0,52.0,0.000060,0.000038
7,20260526_mid_12,SHP + CSV,OK,NaN,41.0,41.0,41.0,41.0,0.000064,0.000042
8,20260527_high_33,SHP + CSV,OK,NaN,38.0,38.0,38.0,38.0,0.000064,0.000040
9,20260527_low_77,SHP + CSV,OK,NaN,56.0,56.0,56.0,56.0,0.000068,0.000036



-----------------------------------
TOPCON REBUILD SUMMARY
-----------------------------------
Successful nodes: 21
Failed nodes: 7
Total rebuilt features: 868.0


In [19]:
failed_stems = [
    "20260506_low_41",
    "20260519_high_35",
    "20260602_low_43",
    "20260713_A4",
    "20260714_A3",
    "20260723_F4"
]

for stem in failed_stems:

    source_shp = CORRECTED_DIR / f"{stem}.shp"
    rebuilt_shp = TOPCON_REBUILT_DIR / f"{stem}.shp"

    print(f"\nRetrying clean rebuild: {stem}")

    try:

        # ------------------------------------------------------
        # Delete previous partial ArcGIS output
        # ------------------------------------------------------
        if arcpy.Exists(str(rebuilt_shp)):
            arcpy.management.Delete(str(rebuilt_shp))

        # Remove any orphaned sidecars from aborted runs
        for f in TOPCON_REBUILT_DIR.glob(f"{stem}.*"):
            try:
                f.unlink()
            except FileNotFoundError:
                pass
            except PermissionError:
                raise RuntimeError(
                    f"{f.name} is locked. Remove this dataset "
                    "from ArcGIS Pro and close its attribute table."
                )

        # ------------------------------------------------------
        # Create a genuinely fresh shapefile through ArcGIS
        # ------------------------------------------------------
        arcpy.management.CopyFeatures(
            str(source_shp),
            str(rebuilt_shp)
        )

        print("  fresh geometry copied")

        # ------------------------------------------------------
        # Add clean fields
        # ------------------------------------------------------
        add_rebuilt_fields(rebuilt_shp)

        # ------------------------------------------------------
        # Find corresponding CSV from pairs
        # ------------------------------------------------------
        pair = next(
            p for p in pairs
            if p["stem"] == stem
        )

        csv_path = pair["csv"]

        # ------------------------------------------------------
        # Restore CSV attributes
        # ------------------------------------------------------
        stats = populate_rebuilt_attributes(
            rebuilt_shp,
            csv_path,
            tolerance=MATCH_TOLERANCE_M
        )

        print(f"  features: {stats['features']}")
        print(f"  CSV rows: {stats['csv_rows']}")
        print(f"  matched:  {stats['matched']}")
        print(f"  updated:  {stats['updated']}")
        print(
            f"  max dist: "
            f"{stats['max_distance_m']:.6f} m"
        )

        # ------------------------------------------------------
        # Replace failed entry in rebuild_log
        # ------------------------------------------------------
        rebuild_log = [
            r for r in rebuild_log
            if r["stem"] != stem
        ]

        rebuild_log.append({
            "stem": stem,
            "method": "SHP + CSV",
            "status": "OK",
            "features": stats["features"],
            "csv_rows": stats["csv_rows"],
            "matched": stats["matched"],
            "updated": stats["updated"],
            "max_distance_m": stats["max_distance_m"],
            "median_distance_m": stats["median_distance_m"]
        })

    except Exception as e:

        print(f"  ERROR: {e}")


Retrying clean rebuild: 20260506_low_41
  fresh geometry copied
  features: 41
  CSV rows: 42
  matched:  41
  updated:  41
  max dist: 0.000063 m

Retrying clean rebuild: 20260519_high_35
  fresh geometry copied
  features: 51
  CSV rows: 51
  matched:  51
  updated:  51
  max dist: 0.000065 m

Retrying clean rebuild: 20260602_low_43
  fresh geometry copied
  features: 45
  CSV rows: 45
  matched:  45
  updated:  45
  max dist: 0.000067 m

Retrying clean rebuild: 20260713_A4
  fresh geometry copied
  features: 56
  CSV rows: 56
  matched:  56
  updated:  56
  max dist: 0.000065 m

Retrying clean rebuild: 20260714_A3
  fresh geometry copied
  features: 51
  CSV rows: 51
  matched:  51
  updated:  51
  max dist: 0.000069 m

Retrying clean rebuild: 20260723_F4
  fresh geometry copied
  features: 26
  CSV rows: 26
  matched:  26
  updated:  26
  max dist: 0.000069 m


In [20]:
low42_csv = TOPCON_CSV_DIR / "20260709_low_42rtk.csv"
low42_output = TOPCON_REBUILT_DIR / "20260709_low_42.shp"

low42 = read_topcon_csv(low42_csv)

bad_xy = low42[
    low42["X"].isna() |
    low42["Y"].isna()
]

if len(bad_xy):
    display(bad_xy)
    raise RuntimeError(
        "low_42 contains invalid X/Y coordinates."
    )

if arcpy.Exists(str(low42_output)):
    arcpy.management.Delete(str(low42_output))

# Remove any orphaned sidecars from a partial run
for f in TOPCON_REBUILT_DIR.glob("20260709_low_42.*"):
    try:
        f.unlink()
    except FileNotFoundError:
        pass

# Create fresh Z-enabled point shapefile
arcpy.management.CreateFeatureclass(
    out_path=str(TOPCON_REBUILT_DIR),
    out_name="20260709_low_42.shp",
    geometry_type="POINT",
    spatial_reference=SOURCE_SR,
    has_m="DISABLED",
    has_z="ENABLED"
)

arcpy.management.AddField(
    str(low42_output),
    "Plot_ID",
    "TEXT",
    field_length=100
)

arcpy.management.AddField(
    str(low42_output),
    "X",
    "DOUBLE"
)

arcpy.management.AddField(
    str(low42_output),
    "Y",
    "DOUBLE"
)

arcpy.management.AddField(
    str(low42_output),
    "Z",
    "DOUBLE"
)

arcpy.management.AddField(
    str(low42_output),
    "SourceCSV",
    "TEXT",
    field_length=100
)

with arcpy.da.InsertCursor(
    str(low42_output),
    ["SHAPE@", "Plot_ID", "X", "Y", "Z", "SourceCSV"]
) as cursor:

    for _, row in low42.iterrows():

        # Topcon CSV convention:
        # CSV X = northing
        # CSV Y = easting

        point = arcpy.Point(
            float(row["Y"]),
            float(row["X"]),
            float(row["Z"])
        )

        geometry = arcpy.PointGeometry(
            point,
            SOURCE_SR,
            has_z=True
        )

        cursor.insertRow([
            geometry,
            row["Plot_ID"],
            row["X"],
            row["Y"],
            row["Z"],
            low42_csv.name
        ])

low42_count = int(
    arcpy.management.GetCount(
        str(low42_output)
    )[0]
)

print(
    f"Created {low42_output.name}: "
    f"{low42_count} features"
)

Created 20260709_low_42.shp: 48 features


In [21]:
rebuild_log = [
    r for r in rebuild_log
    if r["stem"] != "20260709_low_42"
]

rebuild_log.append({
    "stem": "20260709_low_42",
    "method": "CSV -> geometry",
    "status": "OK",
    "features": low42_count,
    "csv_rows": len(low42),
    "matched": low42_count,
    "updated": low42_count,
    "max_distance_m": 0.0,
    "median_distance_m": 0.0
})

In [22]:
qa = pd.DataFrame(rebuild_log).sort_values("stem")

display(qa)

successful = qa["status"].eq("OK")

print("Successful nodes:", successful.sum())
print("Failed nodes:", (~successful).sum())

if "features" in qa.columns:
    print(
        "Total rebuilt features:",
        int(
            pd.to_numeric(
                qa.loc[successful, "features"],
                errors="coerce"
            ).sum()
        )
    )

,stem,method,status,features,csv_rows,matched,updated,max_distance_m,median_distance_m
21,20260506_low_41,SHP + CSV,OK,41,42,41,41,0.000063,0.000043
0,20260511_low_62,SHP + CSV,OK,51,51,51,51,0.000058,0.000042
1,20260512_low_63,SHP + CSV,OK,36,36,36,36,0.000066,0.000042
2,20260513_low_50,SHP + CSV,OK,46,46,46,46,0.000068,0.000039
3,20260518_low_49,SHP + CSV,OK,37,37,37,37,0.000067,0.000040
22,20260519_high_35,SHP + CSV,OK,51,51,51,51,0.000065,0.000034
4,20260521_mid_16,SHP + CSV,OK,52,52,52,52,0.000060,0.000038
5,20260526_mid_12,SHP + CSV,OK,41,41,41,41,0.000064,0.000042
6,20260527_high_33,SHP + CSV,OK,38,38,38,38,0.000064,0.000040
7,20260527_low_77,SHP + CSV,OK,56,56,56,56,0.000068,0.000036


Successful nodes: 28
Failed nodes: 0
Total rebuilt features: 1186


In [23]:
rebuilt_shps = sorted(
    TOPCON_REBUILT_DIR.glob("*.shp")
)

required_fields = {
    "plot_id",
    "x",
    "y",
    "z",
    "sourcecsv"
}

field_rows = []

for shp in rebuilt_shps:

    fields = {
        f.name.lower()
        for f in arcpy.ListFields(str(shp))
    }

    count = int(
        arcpy.management.GetCount(
            str(shp)
        )[0]
    )

    field_rows.append({
        "shapefile": shp.name,
        "features": count,
        "missing_fields": ", ".join(
            sorted(required_fields - fields)
        ),
        "ok": required_fields.issubset(fields)
    })

topcon_field_qa = pd.DataFrame(field_rows)

display(topcon_field_qa)

print(
    "Topcon shapefiles:",
    len(topcon_field_qa)
)

print(
    "All required fields present:",
    topcon_field_qa["ok"].all()
)

print(
    "Total Topcon features:",
    topcon_field_qa["features"].sum()
)

,shapefile,features,missing_fields,ok
0,20260506_low_41.shp,41,,True
1,20260511_low_62.shp,51,,True
2,20260512_low_63.shp,36,,True
3,20260513_low_50.shp,46,,True
4,20260518_low_49.shp,37,,True
5,20260519_high_35.shp,51,,True
6,20260521_mid_16.shp,52,,True
7,20260526_mid_12.shp,41,,True
8,20260527_high_33.shp,38,,True
9,20260527_low_77.shp,56,,True


Topcon shapefiles: 28
All required fields present: True
Total Topcon features: 1186


In [24]:
csv_counts = []

for csv_path in sorted(
    TOPCON_CSV_DIR.glob("*.csv")
):

    df = read_topcon_csv(csv_path)

    csv_counts.append({
        "stem": normalize_stem(csv_path),
        "csv_rows": len(df)
    })

csv_counts = pd.DataFrame(csv_counts)

shp_counts = []

for shp in rebuilt_shps:

    shp_counts.append({
        "stem": shp.stem,
        "shp_features": int(
            arcpy.management.GetCount(
                str(shp)
            )[0]
        )
    })

shp_counts = pd.DataFrame(shp_counts)

count_check = csv_counts.merge(
    shp_counts,
    on="stem",
    how="outer"
)

count_check["difference"] = (
    count_check["csv_rows"]
    - count_check["shp_features"]
)

display(
    count_check.sort_values("stem")
)

,stem,csv_rows,shp_features,difference
0,20260506_low_41,42,41,1
1,20260511_low_62,51,51,0
2,20260512_low_63,36,36,0
3,20260513_low_50,46,46,0
4,20260518_low_49,37,37,0
5,20260519_high_35,51,51,0
6,20260521_mid_16,52,52,0
7,20260526_mid_12,41,41,0
8,20260527_high_33,38,38,0
9,20260527_low_77,56,56,0


now that those are resolved, lets look at and verify the emlid fields

In [25]:
from pathlib import Path
import arcpy
import pandas as pd

CORRECTED_DIR = Path(
    r"N:\Data02\projects-active\BOPclassification_2025\2026 Data\CORRECTED_SHAPEFILES"
)

TOPCON_REBUILT_DIR = Path(
    r"N:\Data02\projects-active\BOPclassification_2025\2026 Data\TOPCON_REBUILT"
)

# Topcon node stems
topcon_stems = {
    shp.stem
    for shp in TOPCON_REBUILT_DIR.glob("*.shp")
}

# Emlid = corrected shapefiles not represented in rebuilt Topcon
emlid_shps = sorted([
    shp
    for shp in CORRECTED_DIR.glob("*.shp")
    if shp.stem not in topcon_stems
])

print("Emlid shapefiles:", len(emlid_shps))

Emlid shapefiles: 31


In [26]:
field_rows = []

for shp in emlid_shps:

    for f in arcpy.ListFields(str(shp)):

        field_rows.append({
            "shapefile": shp.name,
            "field": f.name,
            "type": f.type,
            "length": f.length
        })

field_df = pd.DataFrame(field_rows)

field_summary = (
    field_df
    .groupby(["field", "type", "length"])
    .size()
    .reset_index(name="n_shapefiles")
    .sort_values(
        ["n_shapefiles", "field"],
        ascending=[False, True]
    )
)

display(field_summary)

,field,type,length,n_shapefiles
20,FID,OID,4,31
37,Shape,Geometry,0,31
0,Antenna ht,String,80,30
1,Author,String,80,30
2,Avg end,String,80,30
3,Avg start,String,80,30
4,Base E,String,80,30
5,Base N,String,80,30
6,Base elev,String,80,30
7,Baseline,String,80,30


In [27]:
from pathlib import Path
import arcpy
import pandas as pd
import shutil

EMLID_REBUILT_DIR = Path(
    r"N:\Data02\projects-active\BOPclassification_2025\2026 Data\EMLID_REBUILT"
)

EMLID_REBUILT_DIR.mkdir(parents=True, exist_ok=True)

emlid_log = []

for shp in emlid_shps:

    stem = shp.stem
    out_shp = EMLID_REBUILT_DIR / f"{stem}.shp"

    print(f"\nRebuilding Emlid: {stem}")

    try:
        # Remove previous derivative output if present
        if arcpy.Exists(str(out_shp)):
            arcpy.management.Delete(str(out_shp))

        for f in EMLID_REBUILT_DIR.glob(f"{stem}.*"):
            try:
                f.unlink()
            except FileNotFoundError:
                pass

        # Fresh geometry copy
        arcpy.management.CopyFeatures(
            str(shp),
            str(out_shp)
        )

        # Add standardized fields
        existing = {
            f.name.lower()
            for f in arcpy.ListFields(str(out_shp))
        }

        if "plot_id" not in existing:
            arcpy.management.AddField(
                str(out_shp),
                "Plot_ID",
                "TEXT",
                field_length=100
            )

        if "x" not in existing:
            arcpy.management.AddField(
                str(out_shp),
                "X",
                "DOUBLE"
            )

        if "y" not in existing:
            arcpy.management.AddField(
                str(out_shp),
                "Y",
                "DOUBLE"
            )

        if "z" not in existing:
            arcpy.management.AddField(
                str(out_shp),
                "Z",
                "DOUBLE"
            )

        # Populate from Emlid source fields
        with arcpy.da.UpdateCursor(
            str(out_shp),
            [
                "Name",
                "Northing",
                "Easting",
                "Elevation",
                "Plot_ID",
                "X",
                "Y",
                "Z"
            ]
        ) as cursor:

            updated = 0

            for row in cursor:

                row[4] = row[0]
                row[5] = float(row[1]) if row[1] not in (None, "") else None
                row[6] = float(row[2]) if row[2] not in (None, "") else None
                row[7] = float(row[3]) if row[3] not in (None, "") else None

                cursor.updateRow(row)
                updated += 1

        count = int(
            arcpy.management.GetCount(str(out_shp))[0]
        )

        emlid_log.append({
            "stem": stem,
            "status": "OK",
            "features": count,
            "updated": updated
        })

        print(f"  features: {count}")
        print(f"  updated:  {updated}")

    except Exception as e:

        print(f"  ERROR: {e}")

        emlid_log.append({
            "stem": stem,
            "status": "ERROR",
            "error": str(e)
        })


Rebuilding Emlid: 20260527_mid_17
  features: 42
  updated:  42

Rebuilding Emlid: 20260528_low_47
  features: 40
  updated:  40

Rebuilding Emlid: 20260601_high_29
  ERROR: ERROR 000852: Cannot add field X to 20260601_high_29
Failed to execute (AddField).


Rebuilding Emlid: 20260602_low_51
  features: 50
  updated:  50

Rebuilding Emlid: 20260603_low_60
  features: 45
  updated:  45

Rebuilding Emlid: 20260603_low_68
  features: 20
  updated:  20

Rebuilding Emlid: 20260604_mid_14
  features: 50
  updated:  50

Rebuilding Emlid: 20260608_mid_22
  features: 50
  updated:  50

Rebuilding Emlid: 20260609_mid_20
  features: 50
  updated:  50

Rebuilding Emlid: 20260610_mid_26
  features: 50
  updated:  50

Rebuilding Emlid: 20260611_low_69
  features: 35
  updated:  35

Rebuilding Emlid: 20260611_low_80
  features: 25
  updated:  25

Rebuilding Emlid: 20260615_high_32
  features: 41
  updated:  41

Rebuilding Emlid: 20260615_mid_10
  features: 40
  updated:  40

Rebuilding Emlid: 202606

In [35]:
emlid_qa = pd.DataFrame(emlid_log).sort_values("stem")

display(emlid_qa)

print(
    "Successful Emlid nodes:",
    (emlid_qa["status"] == "OK").sum()
)

print(
    "Failed Emlid nodes:",
    (emlid_qa["status"] != "OK").sum()
)

if "features" in emlid_qa.columns:
    print(
        "Total Emlid features:",
        int(
            pd.to_numeric(
                emlid_qa.loc[
                    emlid_qa["status"] == "OK",
                    "features"
                ],
                errors="coerce"
            ).sum()
        )
    )

,stem,status,features,updated,error
0,20260527_mid_17,OK,42.0,42.0,NaN
1,20260528_low_47,OK,40.0,40.0,NaN
2,20260601_high_29,ERROR,NaN,NaN,ERROR 000852: Cannot add field X to 20260601_h...
3,20260602_low_51,OK,50.0,50.0,NaN
4,20260603_low_60,OK,45.0,45.0,NaN
5,20260603_low_68,OK,20.0,20.0,NaN
6,20260604_mid_14,OK,50.0,50.0,NaN
7,20260608_mid_22,OK,50.0,50.0,NaN
8,20260609_mid_20,OK,50.0,50.0,NaN
9,20260610_mid_26,OK,50.0,50.0,NaN


Successful Emlid nodes: 29
Failed Emlid nodes: 2
Total Emlid features: 1164


In [37]:
stem = "20260601_high_29"

source_shp = CORRECTED_DIR / f"{stem}.shp"
out_shp = EMLID_REBUILT_DIR / f"{stem}.shp"

print(f"Retrying Emlid: {stem}")

# Remove partial derivative
if arcpy.Exists(str(out_shp)):
    arcpy.management.Delete(str(out_shp))

for f in EMLID_REBUILT_DIR.glob(f"{stem}.*"):
    try:
        f.unlink()
    except FileNotFoundError:
        pass
    except PermissionError:
        raise RuntimeError(
            f"{f.name} is locked in ArcGIS Pro."
        )

# Fresh ArcGIS copy
arcpy.management.CopyFeatures(
    str(source_shp),
    str(out_shp)
)

# Add standardized fields
existing = {
    f.name.lower()
    for f in arcpy.ListFields(str(out_shp))
}

if "plot_id" not in existing:
    arcpy.management.AddField(
        str(out_shp), "Plot_ID", "TEXT", field_length=100
    )

if "x" not in existing:
    arcpy.management.AddField(
        str(out_shp), "X", "DOUBLE"
    )

if "y" not in existing:
    arcpy.management.AddField(
        str(out_shp), "Y", "DOUBLE"
    )

if "z" not in existing:
    arcpy.management.AddField(
        str(out_shp), "Z", "DOUBLE"
    )

# Populate standardized attributes
updated = 0

with arcpy.da.UpdateCursor(
    str(out_shp),
    [
        "Name",
        "Northing",
        "Easting",
        "Elevation",
        "Plot_ID",
        "X",
        "Y",
        "Z"
    ]
) as cursor:

    for row in cursor:

        row[4] = row[0]
        row[5] = float(row[1]) if row[1] not in (None, "") else None
        row[6] = float(row[2]) if row[2] not in (None, "") else None
        row[7] = float(row[3]) if row[3] not in (None, "") else None

        cursor.updateRow(row)
        updated += 1

count = int(
    arcpy.management.GetCount(str(out_shp))[0]
)

print("features:", count)
print("updated: ", updated)

Retrying Emlid: 20260601_high_29


RuntimeError: 20260601_high_29.shp.GEO-SPARE1.28852.34512.sr.lock is locked in ArcGIS Pro.

In [38]:
# --------------------------------------------------------------
# FINAL EMLID OUTPUT QA
# --------------------------------------------------------------

rows = []

for stem in sorted(emlid_stems):

    shp = EMLID_REBUILT_DIR / f"{stem}.shp"

    if not arcpy.Exists(str(shp)):

        rows.append({
            "stem": stem,
            "status": "MISSING",
            "features": None,
            "fields_ok": False
        })

        continue

    fields = {
        f.name.lower()
        for f in arcpy.ListFields(str(shp))
    }

    required = {
        "plot_id",
        "x",
        "y",
        "z"
    }

    rows.append({
        "stem": stem,
        "status": "OK",
        "features": int(
            arcpy.management.GetCount(str(shp))[0]
        ),
        "fields_ok": required.issubset(fields)
    })


emlid_final_qa = pd.DataFrame(rows)

display(
    emlid_final_qa.sort_values("stem")
)

print("\nEmlid nodes:", len(emlid_final_qa))

print(
    "Successful:",
    (emlid_final_qa["status"] == "OK").sum()
)

print(
    "All fields correct:",
    emlid_final_qa["fields_ok"].all()
)

print(
    "Total Emlid features:",
    int(emlid_final_qa["features"].fillna(0).sum())
)

,stem,status,features,fields_ok
0,20260527_mid_17.shp,MISSING,None,False
1,20260528_low_47.shp,MISSING,None,False
2,20260601_high_29.shp,MISSING,None,False
3,20260602_low_51.shp,MISSING,None,False
4,20260603_low_60.shp,MISSING,None,False
5,20260603_low_68.shp,MISSING,None,False
6,20260604_mid_14.shp,MISSING,None,False
7,20260608_mid_22.shp,MISSING,None,False
8,20260609_mid_20.shp,MISSING,None,False
9,20260610_mid_26.shp,MISSING,None,False



Emlid nodes: 30
Successful: 0
All fields correct: False
Total Emlid features: 0


C:\Users\scottfordham\AppData\Local\Temp\ipykernel_34512\922203382.py:64: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  int(emlid_final_qa["features"].fillna(0).sum())


In [39]:
print("EMLID_DIR:")
print(EMLID_DIR)

print("\nCORRECTED_DIR:")
print(CORRECTED_DIR)

print("\nEMLID_REBUILT_DIR:")
print(EMLID_REBUILT_DIR)

print("\nFiles currently in EMLID_REBUILT_DIR:")
for shp in sorted(EMLID_REBUILT_DIR.glob("*.shp")):
    print("  ", shp.name)

print("\nFiles currently in CORRECTED_DIR:")
for shp in sorted(CORRECTED_DIR.glob("*.shp")):
    print("  ", shp.name)

EMLID_DIR:
D:\My Drive\BOP_OCTC_2025\2026 RTK Files\EMLID

CORRECTED_DIR:
N:\Data02\projects-active\BOPclassification_2025\2026 Data\CORRECTED_SHAPEFILES

EMLID_REBUILT_DIR:
N:\Data02\projects-active\BOPclassification_2025\2026 Data\EMLID_REBUILT

Files currently in EMLID_REBUILT_DIR:
   20260527_mid_17.shp
   20260528_low_47.shp
   20260602_low_51.shp
   20260603_low_60.shp
   20260603_low_68.shp
   20260604_mid_14.shp
   20260608_mid_22.shp
   20260609_mid_20.shp
   20260610_mid_26.shp
   20260611_low_69.shp
   20260611_low_80.shp
   20260615_high_32.shp
   20260615_mid_10.shp
   20260616_mid_4.shp
   20260630_mid_18.shp
   20260630_mid_8.shp
   20260701_mid_19.shp
   20260707_mid_5.shp
   20260708_high_34.shp
   20260709_low_42rtk.shp
   20260709_mid_2.shp
   20260713_A1.shp
   20260714_B3.shp
   20260715_A5.shp
   20260716_C1.shp
   20260720_C3.shp
   20260721_D5.shp
   20260722_D3.shp
   20260723_F2.shp
   20260804_high_31.shp

Files currently in CORRECTED_DIR:
   20260506_low_41.

In [40]:
stray = EMLID_REBUILT_DIR / "20260709_low_42rtk.shp"

if arcpy.Exists(str(stray)):
    arcpy.management.Delete(str(stray))

for f in EMLID_REBUILT_DIR.glob("20260709_low_42rtk.*"):
    try:
        f.unlink()
    except FileNotFoundError:
        pass

print("Removed stray low_42rtk from EMLID_REBUILT")

Removed stray low_42rtk from EMLID_REBUILT


In [42]:
stem = "20260601_high_29"

source_shp = CORRECTED_DIR / f"{stem}.shp"

# Deliberately use a brand-new name to bypass the locked derivative
out_shp = EMLID_REBUILT_DIR / f"{stem}_fixed.shp"

print("Source:", source_shp)
print("Output:", out_shp)

if arcpy.Exists(str(out_shp)):
    raise RuntimeError(
        f"{out_shp.name} already exists. "
        "Choose another temporary name."
    )

# --------------------------------------------------------------
# Fresh geometry copy
# --------------------------------------------------------------

arcpy.management.CopyFeatures(
    str(source_shp),
    str(out_shp)
)

print("Fresh geometry copied.")

# --------------------------------------------------------------
# Add standardized fields
# --------------------------------------------------------------

existing = {
    f.name.lower()
    for f in arcpy.ListFields(str(out_shp))
}

if "plot_id" not in existing:
    arcpy.management.AddField(
        str(out_shp),
        "Plot_ID",
        "TEXT",
        field_length=100
    )

if "x" not in existing:
    arcpy.management.AddField(
        str(out_shp),
        "X",
        "DOUBLE"
    )

if "y" not in existing:
    arcpy.management.AddField(
        str(out_shp),
        "Y",
        "DOUBLE"
    )

if "z" not in existing:
    arcpy.management.AddField(
        str(out_shp),
        "Z",
        "DOUBLE"
    )

# --------------------------------------------------------------
# Populate standardized fields
#
# Emlid:
#   Name      -> Plot_ID
#   Northing  -> X
#   Easting   -> Y
#   Elevation -> Z
# --------------------------------------------------------------

updated = 0

with arcpy.da.UpdateCursor(
    str(out_shp),
    [
        "Name",
        "Northing",
        "Easting",
        "Elevation",
        "Plot_ID",
        "X",
        "Y",
        "Z"
    ]
) as cursor:

    for row in cursor:

        row[4] = row[0]

        row[5] = (
            float(row[1])
            if row[1] not in (None, "")
            else None
        )

        row[6] = (
            float(row[2])
            if row[2] not in (None, "")
            else None
        )

        row[7] = (
            float(row[3])
            if row[3] not in (None, "")
            else None
        )

        cursor.updateRow(row)
        updated += 1

count = int(
    arcpy.management.GetCount(str(out_shp))[0]
)

print(f"Features: {count}")
print(f"Updated:  {updated}")

Source: N:\Data02\projects-active\BOPclassification_2025\2026 Data\CORRECTED_SHAPEFILES\20260601_high_29.shp
Output: N:\Data02\projects-active\BOPclassification_2025\2026 Data\EMLID_REBUILT\20260601_high_29_fixed.shp
Fresh geometry copied.
Features: 36
Updated:  36


In [43]:
fields = [
    f.name
    for f in arcpy.ListFields(str(out_shp))
]

print(fields)

with arcpy.da.SearchCursor(
    str(out_shp),
    ["Plot_ID", "X", "Y", "Z"]
) as cursor:

    for i, row in enumerate(cursor):
        print(row)

        if i >= 4:
            break

['FID', 'Shape', 'Name', 'Code', 'Code_desc', 'Easting', 'Northing', 'Elevation', 'Desc_', 'Longitude', 'Latitude', 'Ellips_ht', 'Origin', 'Tilt_angle', 'RMS_E', 'RMS_N', 'Elev_RMS', 'Later_RMS', 'Antenna_ht', 'Solution', 'Corr_type', 'Avg_start', 'Avg_end', 'Samples', 'GDOP', 'Base_E', 'Base_N', 'Base_elev', 'Baseline', 'Mount_pt', 'CS_name', 'GPS', 'GLONASS', 'Galileo', 'BeiDou', 'QZSS', 'Dev_type', 'Dev_serial', 'Author', 'Plot_ID', 'X', 'Y', 'Z']
('20260601_high_29_0.0', 4751021.219, 607725.364, 757.505)
('20260601_high_29_0.1', 4751030.424, 607729.52, 758.025)
('20260601_high_29_0.2', 4751016.174, 607734.261, 757.385)
('20260601_high_29_0.3', 4751011.954, 607721.396, 756.444)
('20260601_high_29_11.0', 4751271.311, 607725.977, 752.799)


In [49]:
# --------------------------------------------------------------
# CLEAN EMLID STEMS
# --------------------------------------------------------------

emlid_stems = set()

for zip_path in EMLID_DIR.glob("*.zip"):

    name = zip_path.name

    # remove .zip
    if name.lower().endswith(".zip"):
        name = name[:-4]

    # remove embedded .shp from names like *.shp.zip
    if name.lower().endswith(".shp"):
        name = name[:-4]

    emlid_stems.add(name)

print("Expected Emlid nodes:", len(emlid_stems))

for stem in sorted(emlid_stems):
    print(stem)

Expected Emlid nodes: 30
20260527_mid_17
20260528_low_47
20260601_high_29
20260602_low_51
20260603_low_60
20260603_low_68
20260604_mid_14
20260608_mid_22
20260609_mid_20
20260610_mid_26
20260611_low_69
20260611_low_80
20260615_high_32
20260615_mid_10
20260616_mid_4
20260630_mid_18
20260630_mid_8
20260701_mid_19
20260707_mid_5
20260708_high_34
20260709_mid_2
20260713_A1
20260714_B3
20260715_A5
20260716_C1
20260720_C3
20260721_D5
20260722_D3
20260723_F2
20260804_high_31


In [50]:
# --------------------------------------------------------------
# FINAL EMLID SHAPEFILE LIST
# --------------------------------------------------------------

final_emlid_shps = []

for stem in sorted(emlid_stems):

    if stem == "20260601_high_29":
        shp = EMLID_REBUILT_DIR / "20260601_high_29_fixed.shp"
    else:
        shp = EMLID_REBUILT_DIR / f"{stem}.shp"

    if not arcpy.Exists(str(shp)):
        raise FileNotFoundError(
            f"Missing Emlid dataset: {shp}"
        )

    final_emlid_shps.append(shp)

print("Final Emlid shapefiles:", len(final_emlid_shps))

for shp in final_emlid_shps:
    print(shp.name)

Final Emlid shapefiles: 30
20260527_mid_17.shp
20260528_low_47.shp
20260601_high_29_fixed.shp
20260602_low_51.shp
20260603_low_60.shp
20260603_low_68.shp
20260604_mid_14.shp
20260608_mid_22.shp
20260609_mid_20.shp
20260610_mid_26.shp
20260611_low_69.shp
20260611_low_80.shp
20260615_high_32.shp
20260615_mid_10.shp
20260616_mid_4.shp
20260630_mid_18.shp
20260630_mid_8.shp
20260701_mid_19.shp
20260707_mid_5.shp
20260708_high_34.shp
20260709_mid_2.shp
20260713_A1.shp
20260714_B3.shp
20260715_A5.shp
20260716_C1.shp
20260720_C3.shp
20260721_D5.shp
20260722_D3.shp
20260723_F2.shp
20260804_high_31.shp


In [44]:
# --------------------------------------------------------------
# FIX TOPCON STANDARDIZED X/Y
#
# Existing rebuilt Topcon fields currently contain:
#   X = Northing
#   Y = Easting
#
# Correct convention:
#   X = Easting
#   Y = Northing
#
# Geometry is NOT changed.
# --------------------------------------------------------------

topcon_shps = sorted(
    TOPCON_REBUILT_DIR.glob("*.shp")
)

for shp in topcon_shps:

    print(f"Fixing Topcon XY: {shp.name}")

    with arcpy.da.UpdateCursor(
        str(shp),
        ["X", "Y"]
    ) as cursor:

        for row in cursor:

            old_x = row[0]   # currently Northing
            old_y = row[1]   # currently Easting

            row[0] = old_y   # X = Easting
            row[1] = old_x   # Y = Northing

            cursor.updateRow(row)

print("\nTopcon X/Y corrected.")

Fixing Topcon XY: 20260506_low_41.shp
Fixing Topcon XY: 20260511_low_62.shp
Fixing Topcon XY: 20260512_low_63.shp
Fixing Topcon XY: 20260513_low_50.shp
Fixing Topcon XY: 20260518_low_49.shp
Fixing Topcon XY: 20260519_high_35.shp
Fixing Topcon XY: 20260521_mid_16.shp
Fixing Topcon XY: 20260526_mid_12.shp
Fixing Topcon XY: 20260527_high_33.shp
Fixing Topcon XY: 20260527_low_77.shp
Fixing Topcon XY: 20260602_low_43.shp
Fixing Topcon XY: 20260603_low_59.shp
Fixing Topcon XY: 20260603_low_76.shp
Fixing Topcon XY: 20260604_mid_21.shp
Fixing Topcon XY: 20260615_low_74.shp
Fixing Topcon XY: 20260616_high_36.shp
Fixing Topcon XY: 20260616_low_65.shp
Fixing Topcon XY: 20260630_high_38.shp
Fixing Topcon XY: 20260709_low_42.shp
Fixing Topcon XY: 20260713_A4.shp
Fixing Topcon XY: 20260714_A3.shp
Fixing Topcon XY: 20260715_A6.shp
Fixing Topcon XY: 20260716_B5.shp
Fixing Topcon XY: 20260720_C2.shp
Fixing Topcon XY: 20260721_D4.shp
Fixing Topcon XY: 20260722_D1.shp
Fixing Topcon XY: 20260723_F4.shp
Fi

In [45]:
test_shp = TOPCON_REBUILT_DIR / "20260511_low_62.shp"

with arcpy.da.SearchCursor(
    str(test_shp),
    ["Plot_ID", "X", "Y", "Z", "SHAPE@XY"]
) as cursor:

    for i, row in enumerate(cursor):

        print(row)

        if i >= 4:
            break

('20260511_low_62_rtk', 564064.436, 4802801.713, 894.38, (564064.4360000027, 4802801.713006518))
('20260511_low_62_0.0', 564110.0361, 4803142.8883, 886.7771, (564110.0360924094, 4803142.888341084))
('20260511_low_62_0.1', 564112.4185, 4803152.68, 886.7214, (564112.4184541244, 4803152.679985993))
('20260511_low_62_0.2', 564120.1102, 4803142.1475, 886.7583, (564120.1102265675, 4803142.147491916))
('20260511_low_62_0.3', 564107.8784, 4803132.8834, 886.9347, (564107.8783644212, 4803132.8834039625))


In [51]:
# --------------------------------------------------------------
# FIX EMLID STANDARDIZED X/Y
#
# Correct:
#   X = Easting
#   Y = Northing
#   Z = Elevation
# --------------------------------------------------------------

for shp in final_emlid_shps:

    print(f"Fixing Emlid XY: {shp.name}")

    with arcpy.da.UpdateCursor(
        str(shp),
        [
            "Easting",
            "Northing",
            "Elevation",
            "X",
            "Y",
            "Z"
        ]
    ) as cursor:

        for row in cursor:

            row[3] = (
                float(row[0])
                if row[0] not in (None, "")
                else None
            )

            row[4] = (
                float(row[1])
                if row[1] not in (None, "")
                else None
            )

            row[5] = (
                float(row[2])
                if row[2] not in (None, "")
                else None
            )

            cursor.updateRow(row)

print("\nEmlid X/Y/Z corrected.")

Fixing Emlid XY: 20260527_mid_17.shp
Fixing Emlid XY: 20260528_low_47.shp
Fixing Emlid XY: 20260601_high_29_fixed.shp
Fixing Emlid XY: 20260602_low_51.shp
Fixing Emlid XY: 20260603_low_60.shp
Fixing Emlid XY: 20260603_low_68.shp
Fixing Emlid XY: 20260604_mid_14.shp
Fixing Emlid XY: 20260608_mid_22.shp
Fixing Emlid XY: 20260609_mid_20.shp
Fixing Emlid XY: 20260610_mid_26.shp
Fixing Emlid XY: 20260611_low_69.shp
Fixing Emlid XY: 20260611_low_80.shp
Fixing Emlid XY: 20260615_high_32.shp
Fixing Emlid XY: 20260615_mid_10.shp
Fixing Emlid XY: 20260616_mid_4.shp
Fixing Emlid XY: 20260630_mid_18.shp
Fixing Emlid XY: 20260630_mid_8.shp
Fixing Emlid XY: 20260701_mid_19.shp
Fixing Emlid XY: 20260707_mid_5.shp
Fixing Emlid XY: 20260708_high_34.shp
Fixing Emlid XY: 20260709_mid_2.shp
Fixing Emlid XY: 20260713_A1.shp
Fixing Emlid XY: 20260714_B3.shp
Fixing Emlid XY: 20260715_A5.shp
Fixing Emlid XY: 20260716_C1.shp
Fixing Emlid XY: 20260720_C3.shp
Fixing Emlid XY: 20260721_D5.shp
Fixing Emlid XY: 202

In [52]:
test_emlid = final_emlid_shps[0]

print(test_emlid.name)

with arcpy.da.SearchCursor(
    str(test_emlid),
    ["Plot_ID", "X", "Y", "Z", "SHAPE@XY"]
) as cursor:

    for i, row in enumerate(cursor):
        print(row)

        if i >= 4:
            break

20260527_mid_17.shp
('20260527_mid_17_0.0', 584810.181, 4782842.609, 924.366, (-115.95621161, 43.19359735))
('20260527_mid_17_0.1', 584812.432, 4782852.378, 924.447, (-115.95618241, 43.19368505))
('20260527_mid_17_0.2', 584820.043, 4782840.825, 923.941, (-115.95609053, 43.19358018))
('20260527_mid_17_0.3', 584808.265, 4782832.873, 924.353, (-115.95623668, 43.19350991))
('20260527_mid_17_0.4', 584800.278, 4782844.331, 924.742, (-115.9563332, 43.19361396))


In [53]:
# --------------------------------------------------------------
# FINAL SOURCE DATASETS
# --------------------------------------------------------------

final_topcon_shps = sorted(
    TOPCON_REBUILT_DIR.glob("*.shp")
)

print("Topcon:", len(final_topcon_shps))
print("Emlid :", len(final_emlid_shps))
print("Total :", len(final_topcon_shps) + len(final_emlid_shps))

Topcon: 28
Emlid : 30
Total : 58


In [54]:
# --------------------------------------------------------------
# EXTRACT Plot_ID / X / Y / Z FROM ALL 58 NODES
# --------------------------------------------------------------

records = []

def extract_points(shp, receiver):

    with arcpy.da.SearchCursor(
        str(shp),
        ["Plot_ID", "X", "Y", "Z"]
    ) as cursor:

        for plot_id, x, y, z in cursor:

            records.append({
                "Plot_ID": plot_id,
                "X": x,
                "Y": y,
                "Z": z,
                "Receiver": receiver,
                "SourceFile": shp.name
            })


for shp in final_topcon_shps:
    extract_points(shp, "TOPCON")

for shp in final_emlid_shps:
    extract_points(shp, "EMLID")


master_df = pd.DataFrame(records)

print("Total source features:", len(master_df))
display(master_df.head())

Total source features: 2386


,Plot_ID,X,Y,Z,Receiver,SourceFile
0,20260506_low_41_rtk,549919.9190,4.808013e+06,862.0000,TOPCON,20260506_low_41.shp
1,Low_41_0.0,549902.4594,4.808041e+06,861.4018,TOPCON,20260506_low_41.shp
2,Low_41_0.1,549904.4535,4.808050e+06,860.3251,TOPCON,20260506_low_41.shp
3,Low_41_0.2,549912.2720,4.808038e+06,860.9529,TOPCON,20260506_low_41.shp
4,Low_41_0.3,549900.4516,4.808031e+06,862.4511,TOPCON,20260506_low_41.shp


In [55]:
print("Null Plot_ID:", master_df["Plot_ID"].isna().sum())
print("Null X:", master_df["X"].isna().sum())
print("Null Y:", master_df["Y"].isna().sum())
print("Null Z:", master_df["Z"].isna().sum())

print("\nReceiver counts:")
print(master_df["Receiver"].value_counts())

print("\nDuplicate Plot_ID values:")
duplicates = master_df[
    master_df["Plot_ID"].duplicated(keep=False)
].sort_values("Plot_ID")

display(duplicates)

Null Plot_ID: 0
Null X: 0
Null Y: 0
Null Z: 0

Receiver counts:
Receiver
EMLID     1200
TOPCON    1186
Name: count, dtype: int64

Duplicate Plot_ID values:


,Plot_ID,X,Y,Z,Receiver,SourceFile
174,20260518_low_49_rtk,547791.200,4795193.158,842.118,TOPCON,20260518_low_49.shp
211,20260518_low_49_rtk,552324.060,4788277.265,913.950,TOPCON,20260519_high_35.shp
314,20260526_mid_12_rtk,594169.297,4782144.107,979.513,TOPCON,20260526_mid_12.shp
355,20260526_mid_12_rtk,598944.835,4765331.599,909.953,TOPCON,20260527_high_33.shp
535,20260603_low_76_rtk,610432.730,4759276.912,868.886,TOPCON,20260603_low_76.shp
563,20260603_low_76_rtk,607337.083,4754537.944,747.187,TOPCON,20260604_mid_21.shp
609,20260603_low_76_rtk,615168.645,4760663.250,929.343,TOPCON,20260615_low_74.shp
1649,20260611_low_69_12.0,554377.561,4775693.511,840.335,EMLID,20260611_low_69.shp
1629,20260611_low_69_12.0,554810.937,4775695.217,837.141,EMLID,20260611_low_69.shp
1650,20260611_low_69_12.1,554380.248,4775703.091,840.298,EMLID,20260611_low_69.shp


In [56]:
MASTER_GDB = (
    r"N:\Data02\projects-active\BOPclassification_2025"
    r"\Field Data (Processed)\2026\SRBOP_PointFeatures"
    r"\SRBOP_PointFeatures.gdb"
)

SOURCE_MASTER = (
    MASTER_GDB +
    r"\SRBOP_2026_Master_NAD83_2011"
)

TARGET_MASTER = (
    MASTER_GDB +
    r"\SRBOP_2026_Master_26911"
)

SOURCE_SR = arcpy.SpatialReference(6340)
TARGET_SR = arcpy.SpatialReference(26911)

In [57]:
if arcpy.Exists(SOURCE_MASTER):
    arcpy.management.Delete(SOURCE_MASTER)

arcpy.management.CreateFeatureclass(
    out_path=MASTER_GDB,
    out_name="SRBOP_2026_Master_NAD83_2011",
    geometry_type="POINT",
    spatial_reference=SOURCE_SR,
    has_m="DISABLED",
    has_z="ENABLED"
)

arcpy.management.AddField(
    SOURCE_MASTER,
    "Plot_ID",
    "TEXT",
    field_length=100
)

arcpy.management.AddField(
    SOURCE_MASTER,
    "X",
    "DOUBLE"
)

arcpy.management.AddField(
    SOURCE_MASTER,
    "Y",
    "DOUBLE"
)

arcpy.management.AddField(
    SOURCE_MASTER,
    "Z",
    "DOUBLE"
)

arcpy.management.AddField(
    SOURCE_MASTER,
    "Receiver",
    "TEXT",
    field_length=10
)

arcpy.management.AddField(
    SOURCE_MASTER,
    "SourceFile",
    "TEXT",
    field_length=100
)

<Result 'N:\\Data02\\projects-active\\BOPclassification_2025\\Field Data (Processed)\\2026\\SRBOP_PointFeatures\\SRBOP_PointFeatures.gdb\\SRBOP_2026_Master_NAD83_2011'>

In [58]:
fields = [
    "SHAPE@",
    "Plot_ID",
    "X",
    "Y",
    "Z",
    "Receiver",
    "SourceFile"
]

with arcpy.da.InsertCursor(
    SOURCE_MASTER,
    fields
) as cursor:

    for _, row in master_df.iterrows():

        if pd.isna(row["X"]) or pd.isna(row["Y"]):
            continue

        point = arcpy.Point(
            float(row["X"]),
            float(row["Y"]),
            float(row["Z"]) if not pd.isna(row["Z"]) else None
        )

        geom = arcpy.PointGeometry(
            point,
            SOURCE_SR,
            has_z=True
        )

        cursor.insertRow([
            geom,
            row["Plot_ID"],
            row["X"],
            row["Y"],
            row["Z"],
            row["Receiver"],
            row["SourceFile"]
        ])

print(
    "Source master features:",
    arcpy.management.GetCount(SOURCE_MASTER)[0]
)

Source master features: 2386


In [59]:
if arcpy.Exists(TARGET_MASTER):
    arcpy.management.Delete(TARGET_MASTER)

arcpy.management.Project(
    in_dataset=SOURCE_MASTER,
    out_dataset=TARGET_MASTER,
    out_coor_system=TARGET_SR
)

print(
    "Projected master features:",
    arcpy.management.GetCount(TARGET_MASTER)[0]
)

Projected master features: 2386
